# 02 · Career model: the banker's real balance sheet
Proposal sections 3.1 and 3.2. The client's career is a large, non-tradable asset:

$$R_H - r_f = \beta_B (R_B - r_f) + \beta_S (R_S - r_f) + \varepsilon$$

This notebook shows (1) how big the career is, (2) how the balance sheet changes with age (Figure 1 of the proposal),
(3) what the career behaves like, and (4) why the optimizer will hit its constraints for a young client.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import career as c
from data_loader import load_excess_returns
from personas import BANKER, CIVIL_SERVANT

p = BANKER
pd.Series(asdict(p), name=p.name).to_frame()

## 1. How big is the career?

In [ ]:
bs = c.balance_sheet(p)
summary = pd.Series({
    "discount rate for H": f"{c.discount_rate(p):.2%}",
    "human capital H today": f"CHF {bs['H'].iloc[0]:,.0f}",
    "pension P today": f"CHF {bs['P'].iloc[0]:,.0f}",
    "portfolio F today": f"CHF {bs['F'].iloc[0]:,.0f}",
    "omega = (H+P)/W today": f"{bs['omega'].iloc[0]:.2f}",
    "pension at 65": f"CHF {bs['P'].iloc[-1]:,.0f}",
})
summary.to_frame("value")

## 2. The balance sheet over the working life (Figure 1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(bs.index, bs["H"] / 1e6, bs["P"] / 1e6, bs["F"] / 1e6,
             labels=["Career H", "Pension P", "Portfolio F"], alpha=0.85)
ax.set_xlabel("age"); ax.set_ylabel("CHF million"); ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(bs.index, bs["omega"], color="black", lw=2, label="omega (non-tradable share)")
ax2.set_ylim(0, 1); ax2.set_ylabel("omega"); ax2.legend(loc="upper right")
ax.set_title(f"{p.name}: career turns into pension and portfolio")
fig.tight_layout()

**Reading:** at 35 the career is more than 90% of total wealth, so the portfolio is a small add-on to a huge
bond-and-bank position he already owns. Omega falls with age, which is why the optimal portfolio must change with age (Phase 2).

## 3. What does the career behave like?

In [ ]:
X = load_excess_returns()
RH = c.career_excess_returns(X, p).dropna()
stats = pd.DataFrame({
    "ann. vol": X[["chf_bonds", "eu_banks", "world_eq", "smi"]].std() * np.sqrt(12),
    "corr with career": X[["chf_bonds", "eu_banks", "world_eq", "smi"]].corrwith(RH),
})
print(f"Career (beta_B={p.beta_B}, beta_S={p.beta_S}): ann. vol {RH.std()*np.sqrt(12):.1%}")
stats.style.format("{:.2f}")

In [ ]:
sector_part = (p.beta_S * X[p.sector_asset]).dropna()
bond_part = (p.beta_B * X["chf_bonds"]).dropna()
fig, ax = plt.subplots(figsize=(11, 4))
bond_part.rolling(12).sum().plot(ax=ax, label="bond part (12m)")
sector_part.rolling(12).sum().plot(ax=ax, label="sector part (12m)")
ax.axhline(0, color="black", lw=0.8); ax.legend(); ax.set_title("Where the career's value changes come from")

**Reading:** most month-to-month moves come from interest rates (the bond part, because beta_B is about 3),
but in banking crises (2008, 2011, 2020) the sector part dominates. That is why the backtest measures bad months
on the **sector part** (proposal 4.1).

## 4. The hedge, and why the constraints will bind

In [ ]:
CORE = ["world_eq", "smi", "eu_banks", "ubs", "chf_bonds", "global_bonds_hedged", "swiss_re", "gold"]
Sigma = X[CORE].dropna().cov()
sigma_H = c.career_covariance(Sigma, p)
check = pd.DataFrame({"sigma_H (monthly)": sigma_H,
                      "Sigma^-1 sigma_H": np.linalg.solve(Sigma.values, sigma_H.values)}, index=CORE)
check.round(5)   # second column = beta_B on chf_bonds, beta_S on eu_banks, 0 elsewhere

In [ ]:
rows = {}
for age in [35, 45, 55, 64]:
    b = bs.loc[age]
    h = c.hedging_demand_unconstrained(p, b["omega_H"], b["omega"], CORE)
    rows[age] = {"omega": b["omega"], "hedge in CHF bonds": h["chf_bonds"], "hedge in EU banks": h["eu_banks"]}
pd.DataFrame(rows).T.style.format({"omega": "{:.2f}", "hedge in CHF bonds": "{:+.1%}", "hedge in EU banks": "{:+.1%}"})

**Reading:** at 35 the unconstrained model wants to be *short* thousands of percent of the portfolio in bonds and banks,
which a long-only, no-leverage client cannot do. The optimizer will put these at **zero** (corner solution) and the
shadow prices will measure what that costs. The hedge shrinks with age and becomes feasible near retirement.
This is the result announced in proposal 3.2, not a bug.

## 5. Sensitivity: how much do the assumptions matter?

In [ ]:
betas_S = [0.0, 0.15, 0.3, 0.6, 1.0]
growths = [0.0, 0.01, 0.02, 0.03]
H = pd.DataFrame({f"g={g:.0%}": [c.human_capital(p.with_(beta_S=bS, salary_growth=g)) / 1e6 for bS in betas_S]
                  for g in growths}, index=pd.Index(betas_S, name="beta_S"))
H.style.format("{:.2f}").set_caption("Human capital H today, CHF million")

In [ ]:
cmp = pd.DataFrame({q.name: c.balance_sheet(q)["omega"] for q in (BANKER, CIVIL_SERVANT)})
cmp.plot(figsize=(9, 3.5), title="omega by age: banker vs civil servant", ylim=(0, 1))